In [5]:
from comptox_ai.db import GraphDB
import pandas as pd

In [6]:
db = GraphDB(hostname="neo4j.comptox.ai")

In [65]:
feature_names = [
    'ISOTOPE',
    'Atomic num > 103',
    'Group IVa,Va,VIa Rows 4-6',
    'actinide',
    'Group IIIB,IVB (Sc...)',
    'Lanthanide',
    'Group VB,VIB,VIIB',
    'QAAA@1',
    'Group VIII (Fe...)',
    'Group IIa (Alkaline earth)',
    '4M Ring',
    'Group IB,IIB (Cu..)',
    'ON(C)C',
    'S-S',
    'OC(O)O',
    'QAA@1',
    'CTC',
    'Group IIIA (B...)',
    '7M Ring',
    'Si',
    'C=C(Q)Q',
    '3M Ring',
    'NC(O)O',
    'N-O',
    'NC(N)N',
    'C$=C($A)$A',
    'I',
    'QCH2Q',
    'P',
    'CQ(C)(C)A',
    'QX',
    'CSN',
    'NS',
    'CH2=A',
    'Group IA (Alkali Metal)',
    'S Heterocycle',
    'NC(O)N',
    'NC(C)N',
    'OS(O)O',
    'S-O',
    'CTN',
    'F',
    'QHAQH',
    'OTHER',
    'C=CN',
    'BR',
    'SAN',
    'OQ(O)O',
    'CHARGE',
    'C=C(C)C',
    'CSO',
    'NN',
    'QHAAAQH',
    'QHAAQH',
    'OSO',
    'ON(O)C',
    'O Heterocycle',
    'QSQ','Snot%A%A',
    'S=O',
    'AS(A)A',
    'A$!A$A',
    'N=O',
    'A$A!S',
    'C%N',
    'CC(C)(C)A',
    'QS',
    'QHQH (&...)',
    'QQH',
    'QNQ',
    'NO',
    'OAAO',
    'S=A',
    'CH3ACH3',
    'A!N$A',
    'C=C(A)A',
    'NAN',
    'C=N',
    'NAAN',
    'NAAAN',
    'SA(A)A',
    'ACH2QH',
    'QAAAA@1',
    'NH2',
    'CN(C)C',
    'CH2QCH2',
    'X!A$A',
    'S',
    'OAAAO',
    'QHAACH2A',
    'QHAAACH2A',
    'OC(N)C',
    'QCH3',
    'QN',
    'NAAO',
    '5 M ring',
    'NAAAO',
    'QAAAAA@1',
    'C=C','ACH2N',
    '8M Ring or larger',
    'QO',
    'CL',
    'QHACH2A',
    'A$A($A)$A',
    'QA(Q)Q',
    'XA(A)A',
    'CH3AAACH2A',
    'ACH2O',
    'NCO',
    'NACH2A',
    'AA(A)(A)A',
    'Onot%A%A',
    'CH3CH2A',
    'CH3ACH2A',
    'CH3AACH2A',
    'NAO',
    'ACH2CH2A > 1',
    'N=A',
    'Heterocyclic atom > 1 (&...)',
    'N Heterocycle',
    'AN(A)A',
    'OCO',
    'QQ',
    'Aromatic Ring > 1',
    'A!O!A','A$A!O > 1 (&...)',
    'ACH2AAACH2A',
    'ACH2AACH2A',
    'QQ > 1 (&...)',
    'QH > 1',
    'OACH2A', 
    'A$A!N',
    'X (HALOGEN)',
    'Nnot%A%A',
    'O=A>1', 
    'Heterocycle',
    'QCH2A>1 (&...)',
    'OH',
    'O > 3 (&...)',
    'CH3 > 2  (&...)',
    'N > 1','A$A!O',
    'Anot%A%Anot%A',
    '6M ring > 1',
    'O > 2',
    'ACH2CH2A',
    'AQ(A)A',
    'CH3 > 1',
    'A!A$A!A',
    'NH',
    'OC(C)C',
    'QCH2A',
    'C=O',
    'A!CH2!A',
    'NA(A)A',
    'C-O',
    'C-N',
    'O>1',
    'CH3',
    'N',
    'Aromatic',
    '6M Ring',
    'O',
    'Ring',
    'Fragments',
]

In [88]:
def process_res(chem_result, label):
    dtxsids = []
    maccs_list = []
    for c in chem_result:
        dtxsids.append(c['c']['xrefDTXSID'])
        maccs_list.append([int(bit) for bit in c['c']['maccs']])
    labels = [label for _ in chem_result]
    res_df = pd.DataFrame(maccs_list)
    res_df.columns = feature_names
    res_df.index = dtxsids
    res_df['target'] = labels
    return res_df

def make_qsar_dataset(db_obj, assay, chem_list="PFASMASTER"):
    print(assay)
    active_query = f"MATCH (a:Assay {{ commonName: \"{assay}\" }})<-[r:CHEMICALHASACTIVEASSAY]-(c:Chemical) WHERE c.maccs IS NOT NULL RETURN c;"
    inactive_query = f"MATCH (a:Assay {{ commonName: \"{assay}\" }})<-[r:CHEMICALHASINACTIVEASSAY]-(c:Chemical) WHERE c.maccs IS NOT NULL RETURN c;"

    res_active = db_obj.run_cypher(active_query)
    res_inactive = db_obj.run_cypher(inactive_query)

    try:
        active_df = process_res(res_active, 1)
        inactive_df = process_res(res_inactive, 0)
    except:
        print("  Error - couldn't finish")
        return

    full_df = pd.concat((active_df, inactive_df)).sort_index()

    full_df.to_csv(f"./output/andrea_{assay}.csv", index=True)

    return full_df

In [89]:
assays = [
    'tox21-ar-bla-agonist-p1',
    'tox21-ar-bla-antagonist-p1',
    'tox21-ar-mda-kb2-luc-agonist-p1',
    'tox21-ar-mda-kb2-luc-agonist-p3',
    'tox21-ar-mda-kb2-luc-antagonist-p1',
    'tox21-ar-mda-kb2-luc-antagonist-p2',
    'tox21-er-bla-agonist-p2',
    'tox21-er-bla-antagonist-p1',
    'tox21-er-luc-bg1-4e2-agonist-p2',
    'tox21-er-luc-bg1-4e2-agonist-p4',
    'tox21-er-luc-bg1-4e2-antagonist-p1',
    'tox21-er-luc-bg1-4e2-antagonist-p2',
    'tox21-erb-bla-antagonist-p1',
    'tox21-erb-bla-p1',
    'tox21-err-p1',
    'tox21-pparg-bla-agonist-p1',
    'tox21-pparg-bla-antagonist-p1'
]

In [90]:
for a in assays:
    make_qsar_dataset(db, a)

tox21-ar-bla-agonist-p1
tox21-ar-bla-antagonist-p1
tox21-ar-mda-kb2-luc-agonist-p1
tox21-ar-mda-kb2-luc-agonist-p3
tox21-ar-mda-kb2-luc-antagonist-p1
tox21-ar-mda-kb2-luc-antagonist-p2
tox21-er-bla-agonist-p2
tox21-er-bla-antagonist-p1
tox21-er-luc-bg1-4e2-agonist-p2
tox21-er-luc-bg1-4e2-agonist-p4
tox21-er-luc-bg1-4e2-antagonist-p1
tox21-er-luc-bg1-4e2-antagonist-p2
tox21-erb-bla-antagonist-p1
tox21-erb-bla-p1
tox21-err-p1
  Error - couldn't finish
tox21-pparg-bla-agonist-p1
tox21-pparg-bla-antagonist-p1


,ISOTOPE,Atomic num > 103,"Group IVa,Va,VIa Rows 4-6",actinide,"Group IIIB,IVB (Sc...)",Lanthanide,"Group VB,VIB,VIIB",QAAA@1,Group VIII (Fe...),Group IIa (Alkaline earth),...,C-N,O>1,CH3,N,Aromatic,6M Ring,O,Ring,Fragments,target
DTXSID0020020,0,0,0,0,0,0,0,0,0,0,...,1,1,1,1,1,1,1,1,0,0
DTXSID0020024,0,0,0,0,0,0,0,0,0,0,...,0,1,1,0,0,0,1,0,0,0
DTXSID0020070,0,0,0,0,0,0,0,0,0,0,...,1,1,0,1,0,0,1,0,0,0
DTXSID0020072,0,0,0,0,0,0,0,0,0,0,...,1,0,0,1,1,1,0,1,1,0
DTXSID0020074,0,0,0,0,0,0,0,0,0,0,...,1,1,0,1,0,1,1,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
DTXSID9058600,0,0,0,0,0,0,0,0,0,0,...,0,1,1,0,1,1,1,1,0,0
DTXSID90858850,0,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,1,1,1,0,1
DTXSID90872278,0,0,0,0,0,0,0,0,0,0,...,0,1,1,0,1,1,1,1,0,0
DTXSID90872293,0,0,0,0,0,0,0,0,0,0,...,0,1,1,0,0,0,1,1,0,0
